In [18]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path('.')
RAW_DIR = BASE / 'data' / 'raw'
PROC_DIR = BASE / 'data' / 'processed'
REPORT_DIR = BASE / 'reports'
for d in (RAW_DIR, PROC_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

RAW_CSV = RAW_DIR / 'frailty_raw.csv'
PROC_CSV = PROC_DIR / 'frailty_processed.csv'
REPORT_MD = REPORT_DIR / 'findings.md'

Stage 1 - Ingest

In [19]:
def ingest() -> pd.DataFrame:
    raw = pd.DataFrame({
        'Height_in': [65.8, 71.5, 69.4, 68.2, 67.8, 68.7, 69.8, 70.1, 67.9, 66.8],
        'Weight_lb': [112, 136, 153, 142, 144, 123, 141, 136, 112, 120],
        'Age_yr':    [30, 19, 45, 22, 29, 50, 51, 23, 17, 39],
        'Grip_kg':   [30, 31, 29, 28, 24, 26, 22, 20, 19, 31],
        'Frailty':   ['N', 'N', 'N', 'Y', 'Y', 'N', 'Y', 'Y', 'N', 'N'],
    })
    raw.to_csv(RAW_CSV, index=False)
    df = pd.read_csv(RAW_CSV)
    print(f'Ingested {df.shape[0]} rows x {df.shape[1]} cols from {RAW_CSV}')
    return df

df_raw = ingest()
df_raw

Ingested 10 rows x 5 cols from data/raw/frailty_raw.csv


,Height_in,Weight_lb,Age_yr,Grip_kg,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y
5,68.7,123,50,26,N
6,69.8,141,51,22,Y
7,70.1,136,23,20,Y
8,67.9,112,17,19,N
9,66.8,120,39,31,N


In [20]:
print(df_raw.dtypes, '\n')
print('Missing values per column:\n', df_raw.isna().sum(), '\n')
print('Duplicate rows:', df_raw.duplicated().sum())

Height_in    float64
Weight_lb      int64
Age_yr         int64
Grip_kg        int64
Frailty       object
dtype: object 

Missing values per column:
 Height_in    0
Weight_lb    0
Age_yr       0
Grip_kg      0
Frailty      0
dtype: int64 

Duplicate rows: 0


Stage 2 - Process

In [21]:
AGE_LABELS = ['<30', '30–45', '46–60', '>60']

def process(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Unit standardization
    df['Height_m'] = df['Height_in'] * 0.0254
    df['Weight_kg'] = df['Weight_lb'] * 0.45359237

    # Feature engineering
    df['BMI'] = (df['Weight_kg'] / df['Height_m'] ** 2).round(2)
    df['AgeGroup'] = pd.cut(df['Age_yr'],
                            bins=[-np.inf, 30, 46, 61, np.inf],
                            labels=AGE_LABELS, right=False)

    # Categorical -> numeric encoding
    df['Frailty_binary'] = df['Frailty'].str.strip().str.upper().map({'Y': 1, 'N': 0}).astype('int8')
    dummies = pd.get_dummies(df['AgeGroup'], prefix='AgeGroup', prefix_sep='_', dtype='int8')
    dummies = dummies.reindex(columns=[f'AgeGroup_{l}' for l in AGE_LABELS], fill_value=0)
    df = pd.concat([df, dummies], axis=1)
    return df

df = process(df_raw)
df.to_csv(PROC_CSV, index=False)
print('Saved processed data ->', PROC_CSV)
df

Saved processed data -> data/processed/frailty_processed.csv


,Height_in,Weight_lb,Age_yr,Grip_kg,Frailty,Height_m,Weight_kg,BMI,AgeGroup,Frailty_binary,AgeGroup_<30,AgeGroup_30–45,AgeGroup_46–60,AgeGroup_>60
0,65.8,112,30,30,N,1.67132,50.802345,18.19,30–45,0,0,1,0,0
1,71.5,136,19,31,N,1.81610,61.688562,18.70,<30,0,1,0,0,0
2,69.4,153,45,29,N,1.76276,69.399633,22.33,30–45,0,0,1,0,0
3,68.2,142,22,28,Y,1.73228,64.410117,21.46,<30,1,1,0,0,0
4,67.8,144,29,24,Y,1.72212,65.317301,22.02,<30,1,1,0,0,0
5,68.7,123,50,26,N,1.74498,55.791862,18.32,46–60,0,0,0,1,0
6,69.8,141,51,22,Y,1.77292,63.956524,20.35,46–60,1,0,0,1,0
7,70.1,136,23,20,Y,1.78054,61.688562,19.46,<30,1,1,0,0,0
8,67.9,112,17,19,N,1.72466,50.802345,17.08,<30,0,1,0,0,0
9,66.8,120,39,31,N,1.69672,54.431084,18.91,30–45,0,0,1,0,0


Stage 3 - Analyze

In [22]:
from scipy import stats

def analyze(df: pd.DataFrame):
    numeric_cols = ['Height_in', 'Weight_lb', 'Age_yr', 'Grip_kg',
                    'Height_m', 'Weight_kg', 'BMI', 'Frailty_binary']
    summary = df[numeric_cols].agg(['mean', 'median', 'std']).T.round(3)

    r, p = stats.pointbiserialr(df['Frailty_binary'], df['Grip_kg'])
    grip_by_frailty = df.groupby('Frailty')['Grip_kg'].agg(['count', 'mean', 'median', 'std']).round(2)
    return summary, r, p, grip_by_frailty

summary, r, p, grip_by_frailty = analyze(df)
print(summary, '\n')
print(grip_by_frailty, '\n')
print(f'Correlation (Grip_kg vs Frailty_binary): r = {r:.3f}, p = {p:.3f}')

                   mean   median     std
Height_in        68.600   68.450   1.671
Weight_lb       131.900  136.000  14.232
Age_yr           32.500   29.500  12.860
Grip_kg          26.000   27.000   4.522
Height_m          1.742    1.739   0.042
Weight_kg        59.829   61.689   6.455
BMI              19.682   19.185   1.781
Frailty_binary    0.400    0.000   0.516 

         count   mean  median   std
Frailty                            
N            6  27.67    29.5  4.63
Y            4  23.50    23.0  3.42 

Correlation (Grip_kg vs Frailty_binary): r = -0.476, p = 0.164


In [23]:
def strength_label(r):
    a = abs(r)
    return 'very weak' if a < 0.2 else 'weak' if a < 0.4 else 'moderate' if a < 0.6 else 'strong' if a < 0.8 else 'very strong'

direction = 'negative' if r < 0 else 'positive'
mean_Y = grip_by_frailty.loc['Y', 'mean']; mean_N = grip_by_frailty.loc['N', 'mean']

report = f"""# Findings — Frailty & Grip Strength (Question 1)

## I. Summary statistics (numeric columns)

{summary.to_markdown()}

## AgeGroup counts

{df['AgeGroup'].value_counts().reindex(AGE_LABELS).to_frame('count').to_markdown()}

## II. Grip strength ↔ Frailty

{grip_by_frailty.to_markdown()}

- Correlation between `Grip_kg` and `Frailty_binary`: **r = {r:.3f}** (p = {p:.3f}, n = {len(df)}).
- The relationship is **{strength_label(r)} and {direction}**: frail participants (Y) average **{mean_Y:.2f} kg** grip strength
  vs **{mean_N:.2f} kg** for non-frail participants (N), a difference of {mean_N - mean_Y:.2f} kg.
- This agrees with the literature statement that reduced grip strength in females is associated with higher frailty.
- Caveat: with only 10 participants the p-value is {p:.3f}, so the result is {'not ' if p >= 0.05 else ''}statistically significant at α = 0.05;
  a larger sample would be needed to confirm the effect. Correlation also does not imply causation.
"""
REPORT_MD.write_text(report, encoding='utf-8')
print(report)

# Findings — Frailty & Grip Strength (Question 1)

## I. Summary statistics (numeric columns)

|                |    mean |   median |    std |
|:---------------|--------:|---------:|-------:|
| Height_in      |  68.6   |   68.45  |  1.671 |
| Weight_lb      | 131.9   |  136     | 14.232 |
| Age_yr         |  32.5   |   29.5   | 12.86  |
| Grip_kg        |  26     |   27     |  4.522 |
| Height_m       |   1.742 |    1.739 |  0.042 |
| Weight_kg      |  59.829 |   61.689 |  6.455 |
| BMI            |  19.682 |   19.185 |  1.781 |
| Frailty_binary |   0.4   |    0     |  0.516 |

## AgeGroup counts

| AgeGroup   |   count |
|:-----------|--------:|
| <30        |       5 |
| 30–45      |       3 |
| 46–60      |       2 |
| >60        |       0 |

## II. Grip strength ↔ Frailty

| Frailty   |   count |   mean |   median |   std |
|:----------|--------:|-------:|---------:|------:|
| N         |       6 |  27.67 |     29.5 |  4.63 |
| Y         |       4 |  23.5  |     23   |  3.42 |

- 